# 在线交易（Online Algorithm）

本 Notebook 将原文的 Oanda 接口替换为天勤 TqSdk，以螺纹钢演示在线交易流程：加载模型、接收行情、形成 K 线、生成预测、调整持仓，最后平仓退出。

这里的“在线”指持续接收新数据并使用已有模型推理，而不是每收到一条行情就重新训练。下面先用固定 CSV 验证预测逻辑，再接入实时行情与模拟交易。

## 1. 从离线算法到在线算法

此前的离线研究先准备完整数据集，再根据时间序列特征和方向标签训练 AdaBoost 分类器。在线运行时，未来数据尚未到达，算法只能使用当时可见的信息，随着行情更新预测下一根 K 线的方向。

因此，在线交易不仅需要预测模型，还需要将行情处理、K 线更新、信号生成与委托执行衔接起来。本例沿用离线阶段的特征和模型，通过天勤的事件循环驱动这些环节，使同一套预测逻辑能够用于实时场景。

## 2. 加载模型与标准化参数

在线预测必须复用离线训练时的模型和预处理参数，并保持特征含义、排列顺序与时间尺度一致。否则，即使代码能够运行，输入也可能已经偏离模型训练时的定义。

下面加载前一个螺纹钢 notebook 保存的算法对象，先确认模型及其输入结构，再构造在线预测程序。

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

with Path('algorithm.pkl').open('rb') as f:  # 与离线训练 notebook 使用同一个模型文件
    algorithm = pickle.load(f)  # 保存 model、mu、std；同时包含标的、周期、窗口、滞后阶数、特征顺序与测试期起点

model = algorithm['model']
cols = list(model.feature_names_in_)  # 训练时的准确特征顺序，在线输入必须遵循该顺序
print(f"模型：{type(model).__name__}，输入特征数：{len(cols)}")
print('前六个输入：', cols[:6])

模型：AdaBoostClassifier，输入特征数：36
前六个输入： ['return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_4', 'return_lag_5', 'return_lag_6']


输出显示，加载的模型是 AdaBoost 分类器，接收 36 列滞后特征。这说明当前预测接口的输入结构已明确；接下来还需要保证在线生成的每一列都与离线训练保持一致。

## 3. 将预测逻辑封装为在线算法

特征构造和方向预测封装在同目录 `trading_common.py` 的 `MLTrader` 类中，供 CSV 验证、实时交易和后续日志监控共用。下面直接导入；查看具体实现时，可打开这个公共文件。

本例仅演示固定手数下的多空切换，不加入 Kelly 仓位管理。


In [2]:
from trading_common import MLTrader


这套封装将行情特征转换为下一根 K 线的目标持仓，但不负责确认成交。目标仓位表达策略希望持有什么，实际仓位则取决于委托执行结果；两者将在实时交易环节分别处理。

## 4. 验证在线与离线特征的一致性

连接模拟账户前，先使用历史数据验证在线与离线特征构造的一致性：在相同预测时刻，仅使用当时已完成的 K 线，比较两种实现生成的 36 个模型输入。

该验证用于排查特征计算、列顺序、标准化及时间对齐方面的偏差，属于实现正确性检查，不涉及策略盈利能力的评估。

验证分为两层：先核对保存的特征列顺序与模型输入顺序，再用训练配置独立重建离线输入，与在线输入逐项比较。

In [3]:
# 1. 恢复离线特征。
raw = pd.read_csv('KQ_m_SHFE_rb_600s_8000_20261026.csv', index_col='datetime', parse_dates=True)  # 固定快照，不连接交易账户

# 离线参考直接读取训练配置，不借用待验证 trader 的参数。
training_window = algorithm['window']
training_lags = algorithm['lags']
training_features = algorithm['features']
training_cols = algorithm['cols']

# 重建离线特征，仅用于核对。
reference = pd.DataFrame({'close': raw['close']})
reference['return'] = np.log(reference['close'] / reference['close'].shift(1))
reference['volatility'] = reference['return'].rolling(training_window).std()
mean_return = reference['return'].rolling(training_window).mean()
reference['momentum'] = np.sign(mean_return.mask(
    mean_return.abs() <= algorithm['momentum_tol'], 0.0))
reference['sma'] = reference['close'].rolling(training_window).mean()
reference['min'] = reference['close'].rolling(training_window).min()
reference['max'] = reference['close'].rolling(training_window).max()
reference = reference.dropna()

for feature in training_features:
    for lag in range(1, training_lags + 1):
        reference[f'{feature}_lag_{lag}'] = reference[feature].shift(lag)
reference = reference.dropna()

# 选取原测试期的前 12 个时点，并按训练统计量标准化。
# 天勤拉下来的数据包含训练期和测试期，数据起点不等于测试期起点，所以需要记录 test start
# split = int(len(data) * 0.7)
# test_start = data.index[split]
# test start 就是这么来的
test_times = reference.index[reference.index >= algorithm['test_start']][:12]
x_offline = (reference.loc[test_times, training_cols] - algorithm['mu']) / algorithm['std']

In [4]:
# 2. 建立在线特征，每次仅提供预测时刻之前已结束的 K 线。
trader = MLTrader(algorithm)

online_rows = []
for timestamp in test_times: # timestamp 是要预测的那根 K 线的起始时间
    completed_close = raw.loc[raw.index < timestamp, 'close'] # 找到它之前已经结束的 K 线
    x = trader.prepare_features(completed_close)
    x.index = pd.DatetimeIndex([timestamp])  # 用预测时刻标记这一个在线样本
    online_rows.append(x)
x_online = pd.concat(online_rows)

In [5]:
# 核对保存的训练列顺序与模型实际输入顺序。
assert training_cols == list(algorithm['model'].feature_names_in_)

# 按相同时间、相同列顺序比较两组标准化输入。
assert x_online.index.equals(x_offline.index)
assert x_online.columns.equals(x_offline.columns)
np.testing.assert_allclose(x_online.to_numpy(), x_offline.to_numpy(), rtol=1e-10, atol=1e-10)

# 核对通过后展示预测结果；仅预测，不连接账户或提交委托。
signals = trader.model.predict(x_online)
records = pd.DataFrame({
    '预测 K 线起点': test_times,
    '方向': signals,
    '目标手数': signals * trader.volume,
    '输入核对': '一致',
})
print(f'{len(test_times)} 个时点的在线输入与离线输入全部一致；未连接账户、未提交委托。')
records

12 个时点的在线输入与离线输入全部一致；未连接账户、未提交委托。


,预测 K 线起点,方向,目标手数,输入核对
0,2026-07-13 14:30:00,1,1,一致
1,2026-07-13 14:40:00,1,1,一致
2,2026-07-13 14:50:00,1,1,一致
3,2026-07-13 21:00:00,1,1,一致
4,2026-07-13 21:10:00,1,1,一致
5,2026-07-13 21:20:00,-1,-1,一致
6,2026-07-13 21:30:00,1,1,一致
7,2026-07-13 21:40:00,-1,-1,一致
8,2026-07-13 21:50:00,1,1,一致
9,2026-07-13 22:00:00,1,1,一致


本地运行中，所选 12 个预测时点的在线输入均与离线输入一致，说明两种实现使用了相同的特征值与标准化方式。表格记录了对应的预测方向和目标手数，但没有模拟成交或计算盈亏，不能据此判断盈利能力。

## 5. 接入天勤实时行情和模拟委托

验证预测逻辑后，再将它放入实时行情环境。本例使用天勤模拟账户，行情到达后由事件循环检查 K 线是否结束，再将预测结果转为目标持仓。

主力连续行情可以用于生成信号，但委托必须提交到实际交易合约。与此同时，期货交易需要处理开仓和平仓，不能直接照搬外汇交易单位；下面将这些执行细节交给天勤的目标持仓任务。

In [6]:
# 实时交易需要天勤 SDK；CSV 验证只需公共模块中的 MLTrader。
import time
from tqsdk import TargetPosTask
from trading_common import make_api


在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


### 实时循环：接收行情、预测、交易和结束平仓

本节沿用原书的流程：接收行情，等待新的完整 K 线，计算特征并预测方向，再调整持仓；达到指定更新次数后平仓结束。

天勤直接提供更新中的 K 线序列，因此不需要自行收集 tick 并重采样。主力连续用于生成信号，下单时使用当前对应的实际合约。`TargetPosTask` 负责按目标手数开仓、平仓和反手。由于天勤委托通过事件循环执行，结束时仍需调用 `wait_update()` 推进平仓。

`timeout=60` 表示初始化最多等待 60 秒；运行期间连续 60 秒没有报价更新也会退出。退出后尝试平仓，最多再等待 30 秒。`stop=500` 仍表示报价更新次数，不是运行秒数。


In [7]:
def run_online(algorithm, stop=500, timeout=60):
    # 1. 加载算法、连接模拟账户；初始化最多等待 timeout 秒。
    trader = MLTrader(algorithm)
    with make_api() as api:
        quote = api.get_quote(trader.instrument)
        klines = api.get_kline_serial(
            trader.instrument, trader.bar_seconds,
            data_length=max(100, trader.window + trader.lags + 1),
        )
        deadline = time.time() + timeout
        while not (api.is_serial_ready(klines) and quote.underlying_symbol):
            if not api.wait_update(deadline=deadline):
                raise TimeoutError('行情初始化超时，请检查交易时段和网络。')

        # 2. 主力连续对应的实际合约，用于提交委托。
        contract = quote.underlying_symbol
        target_task = TargetPosTask(api, contract)
        position = api.get_position(contract)
        print(f'信号行情：{trader.instrument} | 模拟交易合约：{contract}')

        # 3. 新 K 线出现时预测；连续 timeout 秒没有报价更新就退出。
        updates = 0  # stop 统计报价更新次数，timeout 控制无报价时的等待时间
        deadline = time.time() + timeout
        try:
            while updates < stop:
                if not (api.wait_update(deadline=deadline) and time.time() < deadline):
                    print(f'连续 {timeout} 秒没有报价更新，结束运行。')
                    break
                if api.is_changing(quote):
                    updates += 1
                    deadline = time.time() + timeout
                if api.is_changing(klines.iloc[-1], 'datetime'):
                    target = trader.on_bar(klines.iloc[:-1]['close'])
                    target_task.set_target_volume(target)

        # 4. 结束时尝试平仓，最多等待 30 秒。
        finally:
            print('*** CLOSING OUT ***')
            target_task.set_target_volume(0)
            deadline = time.time() + 30
            while api.wait_update(deadline=deadline):
                if position.pos_long == 0 and position.pos_short == 0:
                    print('当前持仓已清空。')
                    break
            else:
                print(f'平仓确认超时：多 {position.pos_long} 手，空 {position.pos_short} 手，请检查模拟账户。')


In [ ]:
# 主动启用本行后，才会连接实时行情并提交模拟委托。
# 若 500 次更新期间没有结束一根 10 分钟 K 线，可能没有信号和委托，属于正常现象。
run_online(algorithm, stop=500, timeout=60)

信号行情：KQ.m@SHFE.rb | 模拟交易合约：SHFE.rb2701


## 6. 原文示例输出：单独对照交易过程

以下是原文中四笔成交的主要信息，**来自 Oanda 的书中示例**。

| 顺序 | 提示 | 成交方向与数量（外汇单位） | 成交价 | 示例账户余额 |
|---|---|---:|---:|---:|
| 1 | GOING LONG | 买入 100,000 | 1.19030 | 98,507.7425 |
| 2 | GOING SHORT | 卖出 200,000 | 1.19029 | 98,506.8982 |
| 3 | GOING LONG | 买入 200,000 | 1.19050 | 98,489.1692 |
| 4 | CLOSING OUT | 卖出 100,000 | 1.19051 | 98,490.0049 |


首次买入建立多头；第二笔卖出两倍单位，先平多、再开空；第三笔买入两倍单位，先平空、再开多；最后一笔卖出一倍单位，清空剩余多仓。

天勤对应的学习重点相同，也应更多关注持仓状态变化，如**空仓 → 多头 → 空头 → 多头 → 平仓**。手续费、合约乘数及保证金应按期货账户规则理解，而不是照搬外汇字段。

## 7. 本节要点

本节将离线训练好的模型接入实时行情，形成“接收行情 → 等待 K 线结束 → 构造特征 → 预测方向 → 调整持仓”的在线交易流程。在线预测沿用训练时的特征定义和标准化参数，每次只使用已经结束的 K 线，预测下一根 K 线的方向。

在天勤中，行情更新由 wait_update() 驱动，开仓、平仓和反手由 TargetPosTask 根据目标手数执行。达到指定报价更新次数或等待超时后，程序退出行情循环并尝试平仓。

CSV 核对用于确认在线与离线的模型输入一致；实时模拟则进一步检验行情处理与交易执行能否衔接。前者检查预测逻辑的实现，后者检查完整交易流程的运行。
